# Carga Full e Incremental

- ***Carga Full***: Substitui os dados do destino por uma cópia completa da origem.
- ***Carga Incremental***: Identificar os registros novos ou alterados e aplica as mudanças no destino.

### 1. Preparação

In [ ]:
import pandas as pd
from pathlib import Path

PASTA  = Path("data")
PASTA.mkdir(exist_ok=True)

ARQUIVO_ORIGEM = PASTA / "pedidos_origem.xlsx"
ARQUIVO_DESTINO = PASTA / "pedidos_destino.xlsx"

# Criando os registros na origem
pedidos = pd.DataFrame([
    {"id_pedido": 1, "produto": "Hamburguer", "quantidade": 2, "valor_total": 50.00, "data_atualizacao": "2026-11-18 09:00:00"},
    {"id_pedido": 2, "produto": "Pizza", "quantidade": 1, "valor_total": 40.00, "data_atualizacao": "2026-11-18 09:05:00"},
    {"id_pedido": 3, "produto": "Refrigerante", "quantidade": 3, "valor_total": 30.00, "data_atualizacao": "2026-11-18 09:10:00"}
])

pedidos["data_atualizacao"] = pd.to_datetime(pedidos["data_atualizacao"])
pedidos.to_excel(ARQUIVO_ORIGEM, index=False)

print("Arquivo de origem criado com sucesso! Arquivo:", ARQUIVO_ORIGEM)
pedidos

### 2. Carga Full

In [ ]:
df_origem = pd.read_excel(ARQUIVO_ORIGEM)

# A carga Full vai sobrescrever completamente o destino com os dados da origem
df_destino = df_origem.copy()

# Adicionando uma coluna de data e hora da carga
df_destino["data_atualizacao"] = pd.Timestamp.now()

df_destino.to_excel(ARQUIVO_DESTINO, index=False)

# Guardar a data e horas da última atualização
ultima_carga = df_origem["data_atualizacao"].max()

print(f"Registros carregados: {len(df_origem)}")
print(f"Última atualização: {ultima_carga}")  # Pertinente aos registros
print(f"Última carga realizada em: {pd.Timestamp.now()}")  # Pertinente ao processo de carga
display(pd.read_excel(ARQUIVO_DESTINO))

### 3. Simular mudanças na Origem

In [ ]:
df_origem = pd.read_excel(ARQUIVO_ORIGEM)
df_origem["data_atualizacao"] = pd.to_datetime(df_origem["data_atualizacao"])

# Atualizar o registro do pedido 2
df_origem.loc[df_origem["id_pedido"] == 2, "valor_total"] = 35.00
df_origem.loc[df_origem["id_pedido"] == 2, "data_atualizacao"] = pd.Timestamp("2026-11-18 09:15:00")

novo_pedido = pd.DataFrame([{
    "id_pedido": 4, 
    "produto": "Salada", 
    "quantidade": 1, 
    "valor_total": 20.00, 
    "data_atualizacao": pd.Timestamp("2026-11-18 09:20:00")
    }])

df_origem = pd.concat([df_origem, novo_pedido], ignore_index=True)
df_origem.to_excel(ARQUIVO_ORIGEM, index=False)

display(df_origem)


### 4. Carga Incremental

In [ ]:
df_origem = pd.read_excel(ARQUIVO_ORIGEM)
df_origem["data_atualizacao"] = pd.to_datetime(df_origem["data_atualizacao"])

# Buscar apenas registros novos ou que foram atualizados desde a última carga
df_incremental = df_origem[df_origem["data_atualizacao"] > ultima_carga].copy()

print("Registros identificados para carga incremental:", len(df_incremental))
display(df_incremental)

df_destino = pd.read_excel(ARQUIVO_DESTINO)
df_destino["data_atualizacao"] = pd.to_datetime(df_destino["data_atualizacao"])

# Atualizar registros existentes no destino (UPSERT)
df_destino = pd.concat([df_destino, df_incremental], ignore_index=True)
df_destino = (
    df_destino
    .sort_values("data_atualizacao")
    .drop_duplicates(subset="id_pedido", keep="last")
    .sort_values("id_pedido")
)

df_destino.to_excel(ARQUIVO_DESTINO, index=False)

# Atualizar a data e hora da última carga
ultima_carga = df_destino["data_atualizacao"].max()

print(f"Registros carregados: {len(df_destino)}")
print(f"Última atualização: {ultima_carga}") # Pertinente aos registros

display(pd.read_excel(ARQUIVO_DESTINO))

#? df_incremental["data_carga"] = pd.Timestamp.now()
